In [1]:
import random
import time
from enum import StrEnum
from typing import Self
from rich import print, console
from rich.panel import Panel
from rich.table import Table
from rich.prompt import Prompt, IntPrompt

console = console.Console()


class Names(StrEnum):
    JOHN = "John Lennon"
    PAUL = "Paul McCartney"
    GEORGE = "George Harrison"
    RINGO = "Ringo Starr"


class Beetle:
    health_points: int
    name: Names

    def __init__(
        self,
        health_points: int,
        max_hp: int,
        name: Names
    ) -> None:
        self.health_points = health_points
        self.name = name
        self.max_hp = max_hp

    def __eq__(self, other: Self) -> bool:
        return self.health_points == other.health_points

    def __lt__(self, other: Self) -> bool:
        return self.health_points < other.health_points

    def __le__(self, other: Self) -> bool:
        return self.health_points <= other.health_points

    def __str__(self) -> str:
        return f'Beetle(name="{self.name}", hp={self.health_points})'

    def styling(self) -> str:
        if self.name is Names.JOHN:
            return "in Johny style"
        elif self.name is Names.PAUL:
            return "in McCartney style"
        return "without style"

    def is_alive(self) -> bool:
        return self.health_points > 0

    def heal(self, amount: int = 10) -> None:
        self.health_points = min(self.health_points + amount, self.max_hp)

    def take_damage(self, amount: int) -> None:
        self.health_points -= amount


class BeetlesArmy:
    beetles_list: list[Beetle]
    beetles_name: Names
    beetles_max_health_points: int

    def __init__(
        self,
        beetles_name: Names,
        beetles_army_size: int = 20,
        beetles_max_health_points: int = 100,
    ) -> None:
        self.beetles_list: list[Beetle] = []
        self.beetles_name = beetles_name
        self.max_hp = beetles_max_health_points

        for _ in range(beetles_army_size):
            beetle = Beetle(
                health_points=random.randint(1, self.max_hp),
                max_hp=self.max_hp,
                name=self.beetles_name,
            )
            self.beetles_list.append(beetle)

    def alive_beetles(self) -> list[Beetle]:
        return [b for b in self.beetles_list if b.is_alive()]

    def alive_count(self) -> int:
        return len(self.alive_beetles())

    def total_hp(self) -> int:
        return sum(b.health_points for b in self.alive_beetles())

    def random_alive_beetle(self) -> Beetle | None:
        alive = self.alive_beetles()
        return random.choice(alive) if alive else None

    def pretty_status(self, title: str | None = None) -> Panel:
        alive = self.alive_beetles()
        table = Table(title=f"{title or self.beetles_name}  ({len(alive)} alive)")
        table.add_column("Beetle #", justify="right")
        table.add_column("HP", justify="right")
        for idx, beetle in enumerate(alive, 1):
            table.add_row(str(idx), str(beetle.health_points))
        return Panel(table, expand=False)

    def __eq__(self, other: Self) -> bool:
        return self.alive_count() == other.alive_count()

    def __lt__(self, other: Self) -> bool:
        return self.alive_count() < other.alive_count()

    def __gt__(self, other: Self) -> bool:
        return self.alive_count() > other.alive_count()

def fight(army1: BeetlesArmy, army2: BeetlesArmy) -> BeetlesArmy:
    print(Panel.fit("[bold cyan]БИТВА НАЧИНАЕТСЯ![/bold cyan]"))
    attacker, defender = army1, army2
    round_counter = 0

    while army1.alive_beetles() and army2.alive_beetles():
        round_counter += 1
        print(f"\n[bold yellow]--- Раунд {round_counter} ---[/bold yellow]")

        att_beetle = attacker.random_alive_beetle()
        def_beetle = defender.random_alive_beetle()

        if not (att_beetle and def_beetle):
            break

        damage = random.randint(5, 25)
        def_beetle.take_damage(damage)
        print(
          f"{att_beetle.name} [cyan]{att_beetle.styling()}[/cyan] "
          f"бьёт {def_beetle.name} на [red]{damage}[/red] урона"
        )

        if not def_beetle.is_alive():
            print(f"[bold red]{def_beetle.name} пал в бою![/bold red] ({defender.beetles_name} потеряла бойца)")
            att_beetle.heal(10)
            print(f"{att_beetle.name} восполнил здоровье до {att_beetle.health_points}")

        print(army1.pretty_status("Армия 1"))
        print(army2.pretty_status("Армия 2"))

        attacker, defender = defender, attacker
        time.sleep(1)

    winner = army1 if army1.alive_beetles() else army2
    print(Panel.fit(f"[bold green]Победила армия {winner.beetles_name}![/bold green]"
                    f"\nОсталось жуков: {winner.alive_count()}"))
    return winner

def ask_for_army(number: int) -> BeetlesArmy:
    print(f"\n[bold]Создаём Армию {number}[/bold]")
    name_choice = Prompt.ask("Выберите имя жуков", choices=[n.value for n in Names], default=Names.JOHN.value)
    size = IntPrompt.ask("Сколько жуков в армии?", default=10)
    max_hp = IntPrompt.ask("Максимальное здоровье жука?", default=100)
    return BeetlesArmy(beetles_name=Names(name_choice), beetles_army_size=size, beetles_max_health_points=max_hp)

def main() -> None:
    print(Panel.fit("[bold magenta]Добро пожаловать в эмулятор битвы армий жуков![/bold magenta]"))
    army1 = ask_for_army(1)
    army2 = ask_for_army(2)
    print("\n[bold]Начальные армии:[/bold]")
    print(army1.pretty_status("Армия 1"))
    print(army2.pretty_status("Армия 2"))

    if army1 > army2:
        print("Армия 1 больше по численности!")
    elif army1 < army2:
        print("Армия 2 больше по численности!")
    else:
        print("Армии равны по численности!")

    winner = fight(army1, army2)
    print(f"\n[bold]Итог:[/bold] Победила {winner.beetles_name} "
          f"с {winner.alive_count()} выжившими жуками и {winner.total_hp()} суммарным здоровьем.")

if __name__ == "__main__":
    main()

╭────────────────────────────────────────────────╮
│ Добро пожаловать в эмулятор битвы армий жуков! │
╰────────────────────────────────────────────────╯

Создаём Армию 1

Выберите имя жуков [John Lennon/Paul McCartney/George Harrison/Ringo Starr] (John Lennon):

Ringo Starr


Сколько жуков в армии? (10):

...

Paul McCartney восполнил здоровье до 75

╭───────────────────╮
│    Армия 1  (0    │
│      alive)       │
│ ┏━━━━━━━━━━┳━━━━┓ │
│ ┃ Beetle # ┃ HP ┃ │
│ ┡━━━━━━━━━━╇━━━━┩ │
│ └──────────┴────┘ │
╰───────────────────╯

╭───────────────────╮
│    Армия 2  (1    │
│      alive)       │
│ ┏━━━━━━━━━━┳━━━━┓ │
│ ┃ Beetle # ┃ HP ┃ │
│ ┡━━━━━━━━━━╇━━━━┩ │
│ │        1 │ 75 │ │
│ └──────────┴────┘ │
╰───────────────────╯

╭────────────────────────────────╮
│ Победила армия Paul McCartney! │
│ Осталось жуков: 1              │
╰────────────────────────────────╯

Итог: Победила Paul McCartney с 1 выжившими жуками и 75 суммарным здоровьем.